In [1]:
import requests
from bs4 import BeautifulSoup
from pathlib import Path
import os
import json
import re
import time

/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.2.1) or chardet (7.4.3)/charset_normalizer (3.3.2) doesn't match a supported version!
  warnings.warn(


In [ ]:
tr_data = '../data/tesda_training_regulations.json'
if not os.path.exists(tr_data):
    BASE_URL = "https://www.tesda.gov.ph/Download/Training_Regulations"

    headers = {
        "User-Agent": "Mozilla/5.0"
    }

    session = requests.Session()
    session.headers.update(headers)

    # Dictionary: NC name to TESDA data-id
    training_regulations = {}
    page = 1

    while True:
        url = f"{BASE_URL}?page={page}"
        print(f"Scraping page {page}...")

        response = session.get(url)
        response.raise_for_status()

        soup = BeautifulSoup(response.text, "html.parser")

        # Find the table rows
        rows = soup.select("table.table.table-striped tr")

        # If there are no rows, we've reached the end
        if not rows:
            print("No more rows found. Stopping.")
            break

        page_items = 0

        for row in rows:

            # Get the NC name and download link
            name_cell = row.select_one("td.uppercase")
            download_link = row.select_one("a.dlID[data-id]")

            if name_cell is None or download_link is None:
                continue

            # Strip whitespace/newlines and get actual data-id
            name = name_cell.get_text(strip=True)
            data_id = download_link["data-id"]

            # Save
            training_regulations[name] = int(data_id)
            page_items += 1

        page += 1

        # Be polite to the server
        time.sleep(1)

        # SAVE THE THING
    updated_training_regulations = {
        training_regulation: data_id 
        for training_regulation, data_id 
        in training_regulations.items()
        if "(Superseded)" not in training_regulation
    }

    with open(tr_data, "w", encoding="utf-8") as f:
        json.dump(
            updated_training_regulations,
            f,
            indent=4,
            ensure_ascii=False
        )

    print()
    print(f"Total training regulations: {len(training_regulations)}")
else:
    with open(tr_data, 'r', encoding='utf-8') as file:
        training_regulations = json.load(file)

In [6]:
def download_PDF(
        data_id, #int 
        qualification_name, #string
        session,
        BASE_URL = "https://www.tesda.gov.ph",
        PURPOSE = "study",
        COUNTRY_ID = "123",
        EMAIL="your@email.com"
        ):
    """
        Download the Training Regulation PDF based on the 
        data id found by webscraping
    """
    TR_PAGE = f"{BASE_URL}/Download/Training_Regulations"
    session.headers.update({
        "User-Agent": (
            "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/139.0.0.0 Safari/537.36"
        )
    })


    # Load the TESDA page
    response = session.get(TR_PAGE)
    response.raise_for_status()
    print("TESDA page:", response.status_code)


    # Extract the CSRF token
    soup = BeautifulSoup(response.text, "html.parser")
    token_element = soup.find(
        "input",
        {"name": "__RequestVerificationToken"}
    )
    if token_element is None:
        raise RuntimeError(
            "Could not find __RequestVerificationToken"
        )
    token = token_element["value"]


    # Submit the download form
    download_url = f"{BASE_URL}/Download/DownloadTR"
    data = {
        "__RequestVerificationToken": token,
        "Purpose": PURPOSE,
        "CountryId": COUNTRY_ID,
        "Email": EMAIL,
        "D_id": data_id,
    }

    download_response = session.post(
        download_url,
        data=data,
        allow_redirects=True
    )

    # Save the response
    content_type = download_response.headers.get(
        "Content-Type",
        ""
    ).lower()

    safe_qualification_name = re.sub(
        r'[<>:"/\\|?*]',
        '_',
        qualification_name
    )

    output_file = Path(f"../data/TR_PDFs/TR - [{safe_qualification_name}] - #{data_id}.pdf")

    if "application/pdf" in content_type:
        output_file.write_bytes(
            download_response.content
        )
        print(f"PDF saved to: {output_file}")
    else:
        print("The response was not a PDF.")

In [7]:
session = requests.Session()

for training_regulation, data_id in training_regulations.items():
    print(training_regulation)
    download_PDF(data_id, training_regulation, session)

2D Animation NC III
TESDA page: 200
PDF saved to: ../data/TR_PDFs/TR - [2D Animation NC III] - #123.pdf
2D Game Art Development NC III
TESDA page: 200
PDF saved to: ../data/TR_PDFs/TR - [2D Game Art Development NC III] - #464.pdf
3D Animation NC III
TESDA page: 200
PDF saved to: ../data/TR_PDFs/TR - [3D Animation NC III] - #251.pdf
3D Game Art Development NC III
TESDA page: 200
PDF saved to: ../data/TR_PDFs/TR - [3D Game Art Development NC III] - #477.pdf
5-Axis CNC Machine Operation NC III
TESDA page: 200
PDF saved to: ../data/TR_PDFs/TR - [5-Axis CNC Machine Operation NC III] - #1901.pdf
Able Seafarer Deck NC II (II-5)
TESDA page: 200
PDF saved to: ../data/TR_PDFs/TR - [Able Seafarer Deck NC II (II-5)] - #727.pdf
Able Seafarer Engine NC II (III-5)
TESDA page: 200
PDF saved to: ../data/TR_PDFs/TR - [Able Seafarer Engine NC II (III-5)] - #728.pdf
Agricultural Crops Production NC I
TESDA page: 200
PDF saved to: ../data/TR_PDFs/TR - [Agricultural Crops Production NC I] - #49.pdf
Agricult